In [3]:
import numpy as np
import pandas as pd
df = pd.read_excel("daily-rainfall-at-state-level_21.xlsx")
df = df.fillna(0)

C:\Users\vedpa\AppData\Local\Temp\ipykernel_23520\799906173.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### Example Function to get rainfall data

In [32]:
# writing a function to extract statewise data for rainfall

def get_state_data(df, state_code: int):
    return df[df["state_code"] == state_code].copy()

# example usage
kerala_data = get_state_data(df, 32)
maharashtra_data = get_state_data(df, 27)

### Extracting statewise average weekly rainfall

In [8]:
# ensuring correct types
df["state_code"] = df["state_code"].astype(int)
df["date"] = pd.to_datetime(df["date"])


df = df[["state_code", "date", "actual"]]


df = df.sort_values(["state_code", "date"])

# weekly aggregation
weekly_df = (
    df
    .groupby("state_code")
    .resample("W", on="date")["actual"]
    .mean()
    .reset_index()
)


weekly_df.rename(columns={"actual": "weekly_avg_rainfall"}, inplace=True)

# dictionary to map state codes and state names
state_map = {
    5:"Uttarakhand", 18:"Assam", 16:"Tripura", 36:"Telangana",
    2:"Himachal Pradesh", 1:"Jammu And Kashmir", 17:"Meghalaya",
    31:"Lakshadweep", 27:"Maharashtra", 3:"Punjab",
    20:"Jharkhand", 28:"Andhra Pradesh", 12:"Arunachal Pradesh",
    37:"Ladakh", 35:"Andaman And Nicobar Islands", 30:"Goa",
    9:"Uttar Pradesh", 6:"Haryana", 23:"Madhya Pradesh",
    14:"Manipur", 29:"Karnataka", 4:"Chandigarh",
    19:"West Bengal", 8:"Rajasthan", 22:"Chhattisgarh",
    13:"Nagaland", 10:"Bihar", 34:"Puducherry",
    24:"Gujarat", 21:"Odisha", 33:"Tamil Nadu",
    32:"Kerala", 11:"Sikkim", 7:"Delhi", 15:"Mizoram"
}

weekly_df["state_name"] = weekly_df["state_code"].map(state_map)

state_weekly_data = {
    state: group.copy()
    for state, group in weekly_df.groupby("state_name")
}

# example usage
kerala_df = state_weekly_data["Kerala"]
maharashtra_df = state_weekly_data["Maharashtra"]



for state, df_state in state_weekly_data.items():
    filename = state.replace(" ", "_")
    df_state.to_csv(f"{filename}_weekly_rainfall.csv", index=False)

# print(list(state_weekly_data.keys()))
# print(state_weekly_data)


### interaction of statewise rainfall data with infrastrucute risk (calculated in separate notebook) 

In [9]:
df_state_risk= pd.read_excel('state_infrastructure_features.xlsx')
print(df_state_risk.head())
# REI = rainfall * population_at_risk
# DWE = rainfall * density_infra #higher density=faster spread

for state, df_state in state_weekly_data.items():
    
    df_state["REI"] = (
        df_state["weekly_avg_rainfall"] * df_state_risk["Population_actual"] * df_state_risk["infra_risk"]
    )
    
    df_state["DWE"] = (
        df_state["weekly_avg_rainfall"] * df_state_risk["density_infra"]
    )
    
   
    df_state["REI_norm"] = df_state["REI"] / df_state_risk["Population_actual"]

    

    state_weekly_data[state] = df_state

kerala_df = state_weekly_data["Kerala"]
maharashtra_df = state_weekly_data["Maharashtra"]
# print(kerala_df)
# print(maharashtra_df)

                       state  Population_actual  Population_Density  \
0  Andaman & Nicobar Islands             400000           48.490726   
1             Andhra Pradesh           52787000          323.910215   
2          Arunachal Pradesh            1533000           18.306008   
3                      Assam           35043000          446.760499   
4                      Bihar          123083000         1307.127003   

      urban_pct  sanitation_risk  water_risk  infra_risk  density_infra  \
0  4.300000e-33            0.291       0.099       0.195       9.455692   
1  3.527005e-33            0.291       0.099       0.195      63.162492   
2  2.517939e-33            0.291       0.099       0.195       3.569671   
3  1.531832e-33            0.291       0.099       0.195      87.118297   
4  1.211946e-33            0.291       0.099       0.195     254.889766   

   population_at_risk  
0               78000  
1            10293465  
2              298935  
3             6833385  
4 

In [10]:
df_state_risk= pd.read_excel('state_infrastructure_features.xlsx')
df_state_risk.columns = df_state_risk.columns.str.strip()
df_state_risk.rename(columns={"States/UTs": "state"}, inplace=True)

# Ensure state names are consistent (lowercase + strip)
df_state_risk["state"] = df_state_risk["state"].str.lower().str.strip()

# print(df_state_risk.head())



#combining weekly rainfall data 
df_list = []

for state, df_state in state_weekly_data.items():
    
    df_state = df_state.copy()
    
    # Add state column
    df_state["state"] = state.lower().strip()
    
    df_list.append(df_state)

# Combine all states into one dataframe
df_all = pd.concat(df_list, ignore_index=True)

print(df_all.head())

#merging datasets
df_all = df_all.merge(
    df_state_risk,
    on="state",
    how="left"
)


# Debug: check missing values after merge
missing_states = df_all[df_all["Population_actual"].isna()]["state"].unique()
if len(missing_states) > 0:
    print("Missing states after merge:", missing_states)



df_all["REI"] = (
    df_all["weekly_avg_rainfall"] *
    df_all["Population_actual"] *
    df_all["infra_risk"]
)

df_all["DWE"] = (
    df_all["weekly_avg_rainfall"] *
    df_all["density_infra"]
)


df_all["REI_norm"] = (
    df_all["weekly_avg_rainfall"] *
    df_all["infra_risk"]
)


#sanity checks
print("Summary stats:")
print(df_all[["REI", "DWE", "REI_norm"]].describe())

print(" Null values:")
print(df_all[["REI", "DWE", "REI_norm"]].isnull().sum())


#saving final dataset
# df_all.to_excel("final_dengue_features.xlsx", index=False)
print(df_all.head())
print(df_all.columns.tolist())
# print("Final dataset saved as 'final_dengue_features.csv'")

   state_code       date  weekly_avg_rainfall                   state_name  \
0          35 2021-01-10                  0.0  Andaman And Nicobar Islands   
1          35 2021-01-17                  0.0  Andaman And Nicobar Islands   
2          35 2021-01-24                  0.0  Andaman And Nicobar Islands   
3          35 2021-01-31                  0.0  Andaman And Nicobar Islands   
4          35 2021-02-07                  0.0  Andaman And Nicobar Islands   

   REI  DWE  REI_norm                        state  
0  NaN  NaN       NaN  andaman and nicobar islands  
1  NaN  NaN       NaN  andaman and nicobar islands  
2  NaN  NaN       NaN  andaman and nicobar islands  
3  NaN  NaN       NaN  andaman and nicobar islands  
4  NaN  NaN       NaN  andaman and nicobar islands  
Missing states after merge: ['andaman and nicobar islands' 'delhi' 'jammu and kashmir']
Summary stats:
                REI           DWE     REI_norm
count  1.684000e+03   1684.000000  1684.000000
mean   2.786328e

### Filtering out states with missing data (Andaman, Lakshwadeep, J&K, Delhi, ladakh)

In [12]:
remove_codes = [32, 7, 1, 31,37] #states having missing data

df_filtered = df_all[~df_all["state_code"].isin(remove_codes)].copy()

#restting index
df_filtered.reset_index(drop=True, inplace=True)

df_filtered = df_filtered.dropna()
#debugging
print("Filtering complete\n")

# verifying remaining state codes
print("Remaining state codes:")
print(sorted(df_filtered["state_code"].unique()))


# Check dataset shape
print("\nDataset shape after filtering:")
print(df_filtered.shape)

# Preview data
print("\nSample data:")
print(df_filtered)

df_filtered.to_excel("final_features.xlsx", index=False)

print(df_filtered['state_name'])


Filtering complete

Remaining state codes:
[2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 27, 28, 29, 30, 33, 34, 36]

Dataset shape after filtering:
(1526, 16)

Sample data:
      state_code       date  weekly_avg_rainfall      state_name  \
52            28 2021-01-03                7.454  Andhra Pradesh   
53            28 2021-01-10                3.556  Andhra Pradesh   
54            28 2021-01-17                1.231  Andhra Pradesh   
55            28 2021-01-24                3.762  Andhra Pradesh   
56            28 2021-01-31                1.456  Andhra Pradesh   
...          ...        ...                  ...             ...   
1573          19 2021-12-05                4.052     West Bengal   
1574          19 2021-12-12                5.418     West Bengal   
1575          19 2021-12-19                7.643     West Bengal   
1576          19 2021-12-26               10.276     West Bengal   
1577          19 2022-01-02               